# Day 23 — Function calling & structured outputs

Day 19 built the agent *loop*. Today: the exact **wire format** that loop rides on — how a
tool call travels from your schema, through the model, back to your function, and returns —
plus **structured outputs** that constrain the model's JSON to a schema.

Runnable against a faithful mock; real SDK code shown alongside.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | What "function calling" actually is | 4 min |
| 1 | The tool definition | 8 min |
| 2 | The request/response cycle, block by block | 14 min |
| 3 | Parallel tools, tool_choice, strict | 10 min |
| 4 | Structured outputs: schema-constrained JSON | 14 min |
| 5 | The tool_runner helper + guards recap | 7 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import json, re
from dataclasses import dataclass, field
print("ready")

ready


## 0 — What function calling is (4 min)

The model **cannot run code**. "Function calling" / "tool use" means:

1. You describe your functions as JSON schemas and pass them in `tools`.
2. The model, instead of answering, emits a `tool_use` block: *"call `get_weather` with
   `{"city": "Paris"}`"*.
3. **You** run the function and send the result back as a `tool_result` block.
4. The model continues, now with the result in context.

The model never touches your systems — it only proposes calls. Everything runs on your side,
which is exactly why you validate its arguments (Day 19).

## 1 — The tool definition (8 min)

```python
{
  "name": "get_weather",
  "description": "Get the current weather for a city. Use for 'weather', 'temperature', "
                 "'is it raining' style questions.",     # the model picks tools by DESCRIPTION
  "input_schema": {                                       # JSON Schema
    "type": "object",
    "properties": {
      "city": {"type": "string", "description": "city name, e.g. 'Paris'"},
      "units": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}
    },
    "required": ["city"]
  },
  "strict": true      # optional: guarantee input validates the schema exactly
                      # (needs additionalProperties:false + required; not with programmatic calls)
}
```

The **description** is the single most important field — it's how the model decides whether
and when to call the tool. Write it like docs for a junior engineer: what it does, when to use
it, when *not* to.

In [2]:
def get_weather(city, units="celsius"):
    fake = {"Paris": 14, "Tokyo": 19, "Cairo": 33}
    c = fake.get(city)
    if c is None: return {"error": f"no data for {city}"}
    t = c if units == "celsius" else round(c * 9/5 + 32)
    return {"city": city, "temp": t, "units": units, "conditions": "clear"}

def convert_currency(amount, from_ccy, to_ccy):
    rates = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "JPY"): 150.0}
    r = rates.get((from_ccy, to_ccy))
    return {"error": "unknown pair"} if r is None else {"result": round(amount * r, 2), "rate": r}

TOOLS = {
 "get_weather": dict(fn=get_weather, schema={
   "name": "get_weather",
   "description": "Get current weather for a city. Use for weather/temperature questions.",
   "input_schema": {"type": "object", "additionalProperties": False,
     "properties": {"city": {"type": "string"},
                    "units": {"type": "string", "enum": ["celsius", "fahrenheit"]}},
     "required": ["city"]}, "strict": True}),
 "convert_currency": dict(fn=convert_currency, schema={
   "name": "convert_currency",
   "description": "Convert an amount between two ISO currency codes.",
   "input_schema": {"type": "object", "additionalProperties": False,
     "properties": {"amount": {"type": "number"},
                    "from_ccy": {"type": "string"}, "to_ccy": {"type": "string"}},
     "required": ["amount", "from_ccy", "to_ccy"]}, "strict": True}),
}
print(json.dumps([t["schema"] for t in TOOLS.values()], indent=1)[:400], "...")

[
 {
  "name": "get_weather",
  "description": "Get current weather for a city. Use for weather/temperature questions.",
  "input_schema": {
   "type": "object",
   "additionalProperties": false,
   "properties": {
    "city": {
     "type": "string"
    },
    "units": {
     "type": "string",
     "enum": [
      "celsius",
      "fahrenheit"
     ]
    }
   },
   "required": [
    "city"
   ]
  ...


## 2 — The request/response cycle (14 min)

### Turn 1 — you send tools + question

```python
resp = client.messages.create(
    model="claude-opus-5", max_tokens=1024, tools=[t["schema"] for t in TOOLS.values()],
    messages=[{"role": "user", "content": "What's the weather in Paris in fahrenheit?"}])
```

### Turn 1 — model responds with `stop_reason == "tool_use"`

```
resp.stop_reason == "tool_use"
resp.content == [
    TextBlock(text="I'll check the weather in Paris."),          # optional narration
    ToolUseBlock(id="toolu_01A", name="get_weather",
                 input={"city": "Paris", "units": "fahrenheit"})
]
```

### Turn 2 — you run the tool and send a `tool_result` back

```python
messages.append({"role": "assistant", "content": resp.content})   # echo the whole assistant turn
messages.append({"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": "toolu_01A",
     "content": json.dumps({"city": "Paris", "temp": 57, "units": "fahrenheit"})}
]})
resp2 = client.messages.create(model=..., tools=..., messages=messages)
# resp2.stop_reason == "end_turn", resp2.content == [TextBlock("It's 57°F and clear in Paris.")]
```

Let's run the whole cycle against a mock model that emits real `tool_use` blocks.

In [3]:
@dataclass
class TextBlock: text: str; type: str = "text"
@dataclass
class ToolUseBlock:
    id: str; name: str; input: dict; type: str = "tool_use"
@dataclass
class Message:
    content: list; stop_reason: str = "end_turn"; id: str = "msg_x"; model: str = "claude-opus-5"

class MockToolModel:
    # a scripted "model": decides which tool(s) to call based on the last user text, then answers
    def create(self, *, model, max_tokens, messages, tools=None, tool_choice=None, system=None, **kw):
        user = ""
        for m in reversed(messages):
            if m["role"] == "user" and isinstance(m["content"], str):
                user = m["content"]; break
        results = [b for m in messages if m["role"] == "user" and isinstance(m["content"], list)
                   for b in m["content"] if isinstance(b, dict) and b.get("type") == "tool_result"]

        if results:  # we've already run tools -> produce the final answer
            data = [json.loads(r["content"]) for r in results]
            return Message(content=[TextBlock("Answer: " + "; ".join(json.dumps(d) for d in data))],
                           stop_reason="end_turn")

        calls = []
        u = user.lower()
        if "weather" in u or "temp" in u:
            mcity = re.search(r"\bin ([a-z]+)", u)          # "weather in <city>"
            city = (mcity.group(1).capitalize() if mcity
                    else next((c for c in ["Paris", "Tokyo", "Cairo"] if c.lower() in u), "Paris"))
            units = "fahrenheit" if "fahrenheit" in u or "°f" in u else "celsius"
            calls.append(ToolUseBlock(id="toolu_1", name="get_weather", input={"city": city, "units": units}))
        if "convert" in u or re.search(r"\b(usd|eur|jpy)\b", u):
            m = re.search(r"(\d+)\s*(usd|eur|jpy).*(usd|eur|jpy)", u)
            if m:
                calls.append(ToolUseBlock(id="toolu_2", name="convert_currency",
                    input={"amount": float(m.group(1)), "from_ccy": m.group(2).upper(), "to_ccy": m.group(3).upper()}))
        if not calls:
            return Message(content=[TextBlock("I don't need a tool for that. [direct answer]")],
                           stop_reason="end_turn")
        return Message(content=[TextBlock("Let me look that up.")] + calls, stop_reason="tool_use")

client = type("C", (), {"messages": MockToolModel()})()

def validate(name, args):
    schema = TOOLS[name]["schema"]["input_schema"]
    missing = [k for k in schema["required"] if k not in args]
    if missing: raise ValueError(f"{name}: missing {missing}")
    return args

def run_tool_loop(question, max_steps=4, verbose=True):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        resp = client.messages.create(model="claude-opus-5", max_tokens=1024,
                                      tools=[t["schema"] for t in TOOLS.values()], messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        tool_results = []
        for b in resp.content:
            if b.type == "tool_use":
                try:
                    out = TOOLS[b.name]["fn"](**validate(b.name, b.input))
                except Exception as e:
                    out = {"error": str(e)}
                if verbose: print(f"  [{step}] {b.name}({b.input}) -> {out}")
                tool_results.append({"type": "tool_result", "tool_use_id": b.id,
                                     "content": json.dumps(out)})
        messages.append({"role": "user", "content": tool_results})
    return "stopped: max_steps"

print("A:", run_tool_loop("What's the weather in Tokyo in fahrenheit?"))

  [0] get_weather({'city': 'Tokyo', 'units': 'fahrenheit'}) -> {'city': 'Tokyo', 'temp': 66, 'units': 'fahrenheit', 'conditions': 'clear'}
A: Answer: {"city": "Tokyo", "temp": 66, "units": "fahrenheit", "conditions": "clear"}


## 3 — Parallel tools, `tool_choice`, `strict` (10 min)

In [4]:
# Parallel: one assistant turn can contain MULTIPLE tool_use blocks. Run them, return ALL
# results in ONE user message (splitting them trains the model to stop parallelising).
print("multi-tool question:")
print("A:", run_tool_loop("Convert 100 USD to EUR and tell me the weather in Cairo"))

multi-tool question:
  [0] get_weather({'city': 'Cairo', 'units': 'celsius'}) -> {'city': 'Cairo', 'temp': 33, 'units': 'celsius', 'conditions': 'clear'}
  [0] convert_currency({'amount': 100.0, 'from_ccy': 'USD', 'to_ccy': 'EUR'}) -> {'result': 92.0, 'rate': 0.92}
A: Answer: {"city": "Cairo", "temp": 33, "units": "celsius", "conditions": "clear"}; {"result": 92.0, "rate": 0.92}


### `tool_choice` — force or forbid tool use

| Value | Effect |
| ----- | ------ |
| `{"type": "auto"}` (default) | model decides |
| `{"type": "any"}` | must call *some* tool |
| `{"type": "tool", "name": "get_weather"}` | must call *this* tool |
| `{"type": "none"}` | may not call tools |

**Caveat:** forced tool use (`any` / `tool`) returns a **400 on Claude Fable 5.1 / Mythos 5.1**.
Use `{"type": "auto"}` + an instruction naming the tool, `strict: true` for schema-valid args,
or structured outputs (§4) when you only forced a call to get JSON back.

### `strict: true`

Put it as a **top-level field on the tool definition** (not on `tool_choice`). Requires
`additionalProperties: false` and `required`. Guarantees `tool_use.input` validates your
schema exactly — no missing fields, no hallucinated keys, correct types.

In [5]:
# what strict prevents (our validate() is a hand-rolled stand-in for it)
try:
    validate("convert_currency", {"amount": 100, "from_ccy": "USD"})   # missing to_ccy
except ValueError as e:
    print("strict/validate catches:", e)
print("with strict:true the API guarantees this never reaches your function")

strict/validate catches: convert_currency: missing ['to_ccy']
with strict:true the API guarantees this never reaches your function


## 4 — Structured outputs (14 min)

When you want **the model's answer itself** as schema-valid JSON (not a tool call), use
`output_config.format`. The recommended path is `client.messages.parse()` which validates the
response against a Pydantic model automatically.

```python
from pydantic import BaseModel

class Ticket(BaseModel):
    category: str
    priority: str
    needs_human: bool

resp = client.messages.parse(
    model="claude-opus-5", max_tokens=200,
    messages=[{"role": "user", "content": f"Triage: {ticket_text}"}],
    output_config={"format": {"type": "json_schema", "schema": Ticket.model_json_schema()}},
)
ticket: Ticket = resp.parsed_output          # already a validated Ticket instance
```

`output_config.format` is the current field; the old `output_format` parameter is deprecated.
Structured outputs are **incompatible with document citations** (400). Below: a from-scratch
constrained-decoding stand-in + validator so you see what the guarantee is worth.

In [6]:
# minimal JSON-schema validator (what the API enforces server-side)
def validate_json(obj, schema):
    if schema["type"] == "object":
        if not isinstance(obj, dict): return "not an object"
        for k in schema.get("required", []):
            if k not in obj: return f"missing required '{k}'"
        if schema.get("additionalProperties") is False:
            extra = set(obj) - set(schema["properties"])
            if extra: return f"unexpected keys {extra}"
        for k, v in obj.items():
            if k in schema["properties"]:
                err = validate_json(v, schema["properties"][k])
                if err: return f"{k}: {err}"
        return None
    t = schema["type"]
    ok = {"string": str, "number": (int, float), "integer": int, "boolean": bool}[t]
    if not isinstance(obj, ok): return f"expected {t}"
    if "enum" in schema and obj not in schema["enum"]: return f"{obj!r} not in {schema['enum']}"
    return None

TICKET_SCHEMA = {"type": "object", "additionalProperties": False,
  "properties": {"category": {"type": "string", "enum": ["billing", "bug", "account", "other"]},
                 "priority": {"type": "string", "enum": ["low", "medium", "high"]},
                 "needs_human": {"type": "boolean"}},
  "required": ["category", "priority", "needs_human"]}

# a "model" that sometimes returns messy output; structured outputs would prevent this
messy_outputs = [
 '{"category": "billing", "priority": "high", "needs_human": true}',           # valid
 'Here is the JSON: {"category":"bug","priority":"low","needs_human":false}',   # prose wrapper
 '{"category": "urgent", "priority": "high", "needs_human": true}',             # bad enum
 '{"category": "account", "needs_human": true}',                               # missing priority
]
for raw in messy_outputs:
    m = re.search(r"\{.*\}", raw, re.S)
    obj = json.loads(m.group()) if m else None
    err = validate_json(obj, TICKET_SCHEMA) if obj else "no JSON found"
    print(f"{'OK  ' if err is None else 'FAIL'} {err or ''}   <- {raw[:55]}")
print("\n-> structured outputs make rows 2-4 impossible: the model can only emit tokens that")
print("   keep the JSON schema-valid. No regex extraction, no retry loop for format.")

OK      <- {"category": "billing", "priority": "high", "needs_huma
OK      <- Here is the JSON: {"category":"bug","priority":"low","n
FAIL category: 'urgent' not in ['billing', 'bug', 'account', 'other']   <- {"category": "urgent", "priority": "high", "needs_human
FAIL missing required 'priority'   <- {"category": "account", "needs_human": true}

-> structured outputs make rows 2-4 impossible: the model can only emit tokens that
   keep the JSON schema-valid. No regex extraction, no retry loop for format.


## 5 — The `tool_runner` helper + guards recap (7 min)

You wrote the loop by hand (§2, and Day 19). The SDK's `client.beta.messages.tool_runner`
runs it for you while keeping hooks:

```python
from anthropic import beta_tool

@beta_tool
def get_weather(city: str, units: str = "celsius") -> dict:
    "Get current weather for a city."
    return {...}

runner = client.beta.messages.tool_runner(
    model="claude-opus-5", max_tokens=1024, tools=[get_weather],
    messages=[{"role": "user", "content": "weather in Paris?"}])
for message in runner:            # each iteration = one model turn; tools run automatically
    ...                           # inspect / approve / log / modify results here
final = runner.until_done()
```

You still own: `max_steps` equivalent (iteration cap), argument validation *inside* the
function, a tool allowlist, human approval for destructive tools, and a token/dollar budget.
The helper removes the plumbing, not the responsibility (Day 19 §5).

In [7]:
# guarded loop, matching the Day 19 checklist, over the real tool-use wire format
def run_tool_loop_guarded(question, max_steps=5, tool_budget=4, approve=None):
    messages = [{"role": "user", "content": question}]
    calls = 0
    for step in range(max_steps):
        resp = client.messages.create(model="claude-opus-5", max_tokens=1024,
                                      tools=[t["schema"] for t in TOOLS.values()], messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for b in resp.content:
            if b.type != "tool_use": continue
            calls += 1
            if calls > tool_budget:
                return f"ABORTED: tool budget {tool_budget} exceeded"
            if approve and not approve(b.name, b.input):
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": json.dumps({"error": "denied by policy"}), "is_error": True})
                continue
            try: out = TOOLS[b.name]["fn"](**validate(b.name, b.input))
            except Exception as e: out = {"error": str(e)}
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(out)})
        messages.append({"role": "user", "content": results})
    return "stopped: max_steps"

print(run_tool_loop_guarded("weather in Paris and convert 50 EUR to USD"))

Answer: {"city": "Paris", "temp": 14, "units": "celsius", "conditions": "clear"}; {"result": 54.5, "rate": 1.09}


## 6 — Exercises

1. **Tool description matters.** Change `get_weather`'s description to just "weather". Re-run
   the multi-tool question. (In the mock, edit `MockToolModel` to pick tools by description
   keyword-overlap instead of hard-coded strings — show a vague description causes misfires.)
2. **`is_error` result.** Make `get_weather("Atlantis")` return an error. Send it back with
   `"is_error": true` in the `tool_result`. Show the loop continues and the model can recover
   (ask for a different city) rather than crashing.
3. **Structured output for extraction.** Define a `Person` schema (`name`, `age:int`,
   `emails: string[]`) and validate 3 model outputs (one valid, one wrong type, one extra
   key). Confirm your `validate_json` catches each.
4. **Parallel vs sequential.** Give the mock a question needing two tools where the second
   depends on the first's result. Show the model must do two *turns* (can't parallelise
   dependent calls), and count the API calls.
5. **`tool_choice` simulation.** Add `tool_choice={"type": "tool", "name": "convert_currency"}`
   handling to the mock so it always emits that call first. When is forcing a tool useful, and
   what breaks it on Fable 5.1?
6. **Cost of tool schemas.** Every request re-sends all tool schemas. Estimate the token cost
   of 10 tools averaging 80 tokens of schema each, on a 20-turn conversation. What reduces it?
   (Hint: prompt caching, `defer_loading` + tool search.)

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. Can the model run your functions? What does it actually do?
2. Walk the four steps of one tool-use round trip and which role each message has.
3. Why return all parallel tool results in a single message?
4. What does `strict: true` guarantee, and where do you put it?
5. When would you use structured outputs instead of a forced tool call?
6. Which `tool_choice` values return a 400 on Claude Fable 5.1, and what do you do instead?
7. Every request re-sends every tool schema. Name two ways to cut that cost.

## Where this goes next

- **Day 24 — Streaming:** the SSE event stream — `message_start`, `content_block_delta`,
  accumulating text and tool-call JSON as it arrives, and building a console streamer.